## Preparazione Dataset Papers With Code (PWC)
In questa sezione, andremo ad attingere al dataset `pwc-archive/papers-with-abstracts` su Hugging Face.
- Salveremo i dati grezzi
- Ne applicheremo un filtraggio su termini AI tramite white-list regex
- Genereremo due varianti: Multiclasse (con label associate ad applicazioni aziendali e AI) e Multilabel (con le keyword tecniche per i task)

In [ ]:
# 1. Setup ed estrazione dati
%pip install datasets -q

import re
import ast
import pandas as pd
from pathlib import Path
from datasets import load_dataset

RAW_PWC_DIR = Path('../data/raw/pwc-archive')
RAW_PWC_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = Path('../data/processed')

print("Scaricamento del dataset pwc-archive/papers-with-abstracts...")
dataset = load_dataset("pwc-archive/papers-with-abstracts", split="train")

# Salva la variante grezza del dataset in csv per sicurezza e riproducibilità offline
raw_csv_path = RAW_PWC_DIR / 'papers_with_abstracts_raw.csv'
dataset.to_csv(raw_csv_path)
print(f"Dataset salvato localmente su in {raw_csv_path}")

In [ ]:
# 2. Caricamento Dataframe e Selezione Colonne
df_pwc = pd.read_csv(raw_csv_path, low_memory=False)

columns_to_keep = ['title', 'abstract', 'short_abstract', 'tasks', 'methods']
# manteniamo solo le colonne richieste
df_pwc = df_pwc[[col for col in columns_to_keep if col in df_pwc.columns]]

# Gestione missing values con default vuoti
df_pwc['title'] = df_pwc['title'].fillna('')
df_pwc['abstract'] = df_pwc['abstract'].fillna('')
df_pwc['short_abstract'] = df_pwc['short_abstract'].fillna('')
df_pwc['tasks'] = df_pwc['tasks'].fillna('[]')
df_pwc['methods'] = df_pwc['methods'].fillna('[]')

print(f"Dimensione dataset pre-filtro: {len(df_pwc)} righe con le seguenti colonne:")
print(list(df_pwc.columns))

In [ ]:
# 3. Filtraggio record inerenti all'Intelligenza Artificiale
# Definiamo una regex estremamente comprensiva (whitelist)
ai_keywords = [
    r'\bartificial\s+intelligence\b', r'\bai\b(?!\s*[-\.])', r'\bmachine\s+learning\b', r'\bml\b(?!\s*[-\.])',
    r'\bdeep\s+learning\b', r'\bdl\b(?!\s*[-\.])', r'\bneural\s+network[s]?\b', r'\bnlp\b',
    r'\bnatural\s+language\s+processing\b', r'\bcomputer\s+vision\b', r'\bcv\b(?!\s*[-\.])',
    r'\breinforcement\s+learning\b', r'\brl\b', r'\blarge\s+language\s+model[s]?\b',
    r'\bllm[s]?\b', r'\bgenerative\s+ai\b', r'\btransformer[s]?\b', r'\bbert\b', r'\bgpt\b',
    r'\bconvolutional\s+neural\s+network[s]?\b', r'\bcnn[s]?\b', r'\bvision\s+transformer[s]?\b', r'\bvit\b',
    r'\brecurrent\s+neural\s+network[s]?\b', r'\brnn[s]?\b', r'\blstm\b', r'\blong\s+short-term\s+memory\b',
    r'\bgenerative\s+adversarial\s+network[s]?\b', r'\bgan[s]?\b', r'\bautoencode(?:r|rs)\b',
    r'\bvariational\s+autoencoder[s]?\b', r'\bvae[s]?\b', r'\bdiffusion\s+model[s]?\b',
    r'\bgraph\s+neural\s+network[s]?\b', r'\bgnn[s]?\b', r'\bmultilayer\s+perceptron[s]?\b', r'\bmlp\b',
    r'\bsupport\s+vector\s+machine[s]?\b', r'\bsvm[s]?\b', r'\brandom\s+forest[s]?\b', r'\bdecision\s+tree[s]?\b', 
    r'\bxgboost\b', r'\bgradient\s+boosting\b', r'\blightgbm\b', r'\bcatboost\b', r'\bensemble\s+method[s]?\b',
    r'\bk-nearest\s+neighbor[s]?\b', r'\bknn\b', r'\bnaive\s+bayes\b', r'\blogistic\s+regression\b',
    r'\bcluster(?:ing)?\b', r'\bk-means\b', r'\bprincipal\s+component\s+analysis\b', r'\bpca\b',
    r'\bdata\s+science\b', r'\bdata\s+mining\b', r'\brobotic[s]?\b', r'\bexpert\s+system[s]?\b', 
    r'\bpredictive\s+model(?:ing)?\b', r'\bchatbots?\b', r'\bimage\s+recognition\b', r'\bobject\s+detection\b',
    r'\bsemantic\s+segmentation\b', r'\bimage\s+classification\b', r'\bspeech\s+recognition\b', 
    r'\bsentiment\s+analysis\b', r'\bnamed\s+entity\s+recognition\b', r'\bner\b', r'\bword\s+embedding[s]?\b',
    r'\bword2vec\b', r'\bfederated\s+learning\b', r'\btransfer\s+learning\b', r'\bzero-shot\s+learning\b',
    r'\bfew-shot\s+learning\b', r'\bsupervised\s+learning\b', r'\bunsupervised\s+learning\b', r'\bsemi-supervised\s+learning\b',
    r'\bq-learning\b', r'\bdeep\s+q-network[s]?\b', r'\bdqn\b', r'\bknowledge\s+graph[s]?\b',
    r'\bbig\s+data\b'
]
ai_pattern = re.compile('|'.join(ai_keywords), re.IGNORECASE)

def is_ai_related(row):
    # Uniamo testo dalle 5 colonne per avere la masima finestra di ricerca
    text_to_search = ' '.join(str(val) for val in row.values)
    return bool(ai_pattern.search(text_to_search))

print("Esecuzione regex di filtraggio (questa operazione richiederà alcuni istanti)...")
df_ai = df_pwc[df_pwc.apply(is_ai_related, axis=1)].copy()

filtered_csv_path = PROCESSED_DIR / 'pwc_ai_filtered.csv'
df_ai.to_csv(filtered_csv_path, index=False)

print(f"Record identificati come AI: {len(df_ai)} ({len(df_ai)/len(df_pwc):.1%} del totale)")
print(f"Salvato dataset intermedio in: {filtered_csv_path}")

In [ ]:
# 4. Generazione Dataset Multiclasse (description, label)

categories_regex = {
    "Healthcare": [r'\bhealthcare\b', r'\bmedical\s+imag(?:e|ing)\b', r'\bclinical\s+trial[s]?\b', r'\bdisease\s+diagnos[is|tic]\b', r'\bpatient\s+care\b', r'\belectronic\s+health\s+record[s]?\b', r'\behr\b', r'\bprecision\s+medicine\b', r'\bdrug\s+discovery\b', r'\bgenomic[s]?\s+analysis\b', r'\bbioinformatics\b', r'\btelehealth\b', r'\btelemedicine\b', r'\bsurgical\s+roboti(?:cs|c)\b'],
    "Media & Entertainment": [r'\bsocial\s+media\b', r'\bvideo\s+game[s]?\b', r'\bmovie\s+recommendation[s]?\b', r'\bmusic\s+generation\b', r'\bentertainment\s+industry\b', r'\bmultimedia\b', r'\bcontent\s+creation\b', r'\bdeepfake[s]?\b', r'\bfake\s+news\s+detection\b', r'\bgenerative\s+art\b', r'\btext-to-speech\b', r'\bcontent\s+moderation\b'],
    "Fintech": [r'\bfintech\b', r'\bfinancial\s+market[s]?\b', r'\bbanking\s+sector\b', r'\bstock\s+market\b', r'\balgorithmic\s+trading\b', r'\bfraud\s+detection\b', r'\bcredit\s+scoring\b', r'\bcryptocurrency\b', r'\bdecentralized\s+finance\b', r'\brobo-advis(?:or|ing)\b', r'\bquantitative\s+trading\b', r'\brisk\s+assessment\b', r'\banti-money\s+laundering\b', r'\binsurtech\b'],
    "Autonomous Driving & UVs": [r'\bautonomous\s+driv(?:e|ing)\b', r'\bautonomous\s+vehicle[s]?\b', r'\bself[-\s]driving\b', r'\bdriverless\b', r'\bunmanned\s+aerial\s+vehicle[s]?\b', r'\buav[s]?\b', r'\bdrone\s+navigation\b', r'\bavs\b', r'\badas\b', r'\badvanced\s+driver\s+assistance\b', r'\blidar\s+point\s+cloud\b', r'\btrajectory\s+prediction\b', r'\btraffic\s+sign\s+recognition\b'],
    "Robotics & Industry": [r'\bindustrial\s+roboti(?:cs|c)\b', r'\brobotic\s+arm[s]?\b', r'\bsmart\s+manufacturing\b', r'\bsmart\s+factory\b', r'\bsupply\s+chain\s+management\b', r'\bindustry\s+4\.0\b', r'\bpredictive\s+maintenance\b', r'\biot\s+devices\b', r'\bdefect\s+detection\b', r'\bquality\s+inspection\b', r'\bdigital\s+twin[s]?\b', r'\bcollaborative\s+robot[s]?\b', r'\bcobot[s]?\b', r'\bwarehouse\s+automation\b'],
    "Environment": [r'\bclimate\s+change\b', r'\bprecision\s+agriculture\b', r'\benvironmental\s+monitoring\b', r'\bweather\s+forecast(?:ing)?\b', r'\bsustainability\b', r'\brecycl(?:e|ing)\b', r'\brenewable\s+energy\b', r'\bsatellite\s+imagery\b', r'\bsmart\s+grid[s]?\b', r'\bpollutant\s+monitoring\b', r'\bbiodiversity\s+monitoring\b', r'\bcarbon\s+footprint\b'],
    "Virtual Assistants": [r'\bvirtual\s+assistant[s]?\b', r'\bchatbot[s]?\b', r'\bdialog(?:ue)?\s+system[s]?\b', r'\bconversational\s+agent[s]?\b', r'\bvoice\s+assistant[s]?\b', r'\bconversational\s+ai\b', r'\bintent\s+recognition\b', r'\bspoken\s+language\s+understanding\b', r'\bsmart\s+speaker[s]?\b', r'\bautomatic\s+speech\s+recognition\b', r'\bquestion\s+answering\s+system[s]?\b'],
    "Enterprise": [r'\benterprise\s+software\b', r'\bb2b\b', r'\bhuman\s+resources\b', r'\bmarketing\s+automation\b', r'\bcustomer\s+relationship\s+management\b', r'\bcrm\b', r'\be-commerce\b', r'\bbusiness\s+intelligence\b', r'\bchurn\s+prediction\b', r'\bsales\s+forecasting\b', r'\bintelligent\s+document\s+processing\b', r'\brobotic\s+process\s+automation\b', r'\brpa\b', r'\benterprise\s+resource\s+planning\b'],
    "Data Science": [r'\bdata\s+science\b', r'\bexploratory\s+data\s+analysis\b', r'\bdatabase\s+management\b', r'\bdata\s+pipeline[s]?\b', r'\bbig\s+data\s+analytics\b', r'\banomaly\s+detection\b', r'\bdimensionality\s+reduction\b', r'\bfeature\s+engineering\b', r'\bautoml\b', r'\bactive\s+learning\b', r'\bimbalanced\s+learning\b'],
    "Research": [r'\bscientific\s+research\b', r'\bliterature\s+review\b', r'\bacademic\b', r'\bempirical\s+study\b', r'\breproducible\s+research\b', r'\btheoretical\s+analysis\b']
}

compiled_regexes = {cat: re.compile('|'.join(kws), re.IGNORECASE) for cat, kws in categories_regex.items()}

def map_domain_category(text):
    text_lower = str(text).lower()
    for cat, regex in compiled_regexes.items():
        if regex.search(text_lower):
            return cat
    # Categoria di default molto applicata nel target papers-with-code
    return "Research"

# Crea dataset con due sole colonne
df_multiclass = pd.DataFrame()
df_multiclass['description'] = df_ai['abstract']
# Match deterministico basato sul testo (titolo + abstract spesso offrono il contesto dell'applicabilità)
df_multiclass['label'] = (df_ai['title'] + ' ' + df_ai['abstract']).apply(map_domain_category)

# Filtraggio o pulizie opzionali...
multiclass_path = PROCESSED_DIR / 'pwc_ai_multiclass.csv'
df_multiclass.to_csv(multiclass_path, index=False)
print(f"Salvato dataset Multiclasse: {multiclass_path}")
print(df_multiclass['label'].value_counts())

In [ ]:
# 5. Generazione Dataset Multilabel (description, labels)

import numpy as np

def clean_and_extract_tasks(task_val):
    if task_val is None or (isinstance(task_val, float) and pd.isna(task_val)):
        return ''
    
    tasks = []
    
    # Se il dato è già una lista o un array numpy (comune nell'estrazione da HF datasets in pandas)
    if isinstance(task_val, (list, tuple, np.ndarray)):
        parsed_list = task_val
    else:
        val_str = str(task_val).strip()
        if val_str == '[]' or not val_str:
            return ''
            
        # 1. Utilizzo ast.literal_eval per parsare in modo sicuro le stringhe list-like
        try:
            parsed_list = ast.literal_eval(val_str)
        except (ValueError, SyntaxError):
            # 2. Fallback: la stringa è un ndarray "strigato" stampando ['Task A' 'Task B'] senza virgole
            import re
            matches = re.findall(r"'(.*?)'|\"(.*?)\"", val_str)
            if matches:
                # Estrae le stringhe ignorando le stringhe vuote dei gruppi regex
                parsed_list = [m[0] or m[1] for m in matches]
            else:
                cleaned_str = val_str.replace('[', '').replace(']', '').replace("'", '').replace('"', '')
                parsed_list = [t.strip() for t in cleaned_str.split(',')] if cleaned_str else []
            
    if isinstance(parsed_list, (list, tuple, np.ndarray)):
        for item in parsed_list:
            if isinstance(item, dict) and 'task' in item:
                tasks.append(item['task'])
            elif isinstance(item, str):
                tasks.append(item)
            
    # Normalize e deduplicate per ogni record
    unique_tasks = set()
    for t in tasks:
        # Pulisci, togli hyphen in favore di spazio e trasforma a lower case
        normalized = str(t).strip().lower().replace('-', ' ')
        if normalized:
            unique_tasks.add(normalized)
            
    # Ordiniamo alfabeticamente per riproducibilità
    # Utilizziamo il pipe '|' come separatore preferenziale in file CSV
    return '|'.join(sorted(list(unique_tasks)))

df_multilabel = pd.DataFrame()
df_multilabel['description'] = df_ai['abstract']
df_multilabel['labels'] = df_ai['tasks'].apply(clean_and_extract_tasks)

# Opzionale: rimuovi record in cui non avevamo nessun "task" AI indicato dal dataset originale
df_multilabel = df_multilabel[df_multilabel['labels'] != '']

multilabel_path = PROCESSED_DIR / 'pwc_ai_multilabel.csv'
df_multilabel.to_csv(multilabel_path, index=False)

print(f"Salvato dataset Multilabel con etichette normalizzate per i tasks in: {multilabel_path}")
print(f"Record rimanenti usabili: {len(df_multilabel)}")
print("\nPrime righe - Sample labels:")
print(df_multilabel.head(5))

In [ ]:
from IPython.display import display

print("=== Prime 20 righe del Dataset Multiclasse ===")
display(df_multiclass.head(20))

print("\n=== Prime 20 righe del Dataset Multilabel ===")
display(df_multilabel.head(20))